# Lab 07: Production Deployment

## Objectives
- Deploy Spark applications using Docker containers
- Configure production-ready Spark clusters
- Implement Kubernetes deployment with Spark on Kubernetes
- Set up monitoring and logging with Prometheus and Grafana
- Configure CI/CD pipelines with GitHub Actions
- Implement security best practices and secret management
- Configure high availability and fault tolerance
- Optimize performance for production workloads

## Domain Coverage
- **Production — 5%**
- **Spark Architecture — 20%**

## Estimates
- **Time:** ~90-120 minutes
- **Difficulty:** Advanced
- **Prerequisites:** Lab 01 (Spark Fundamentals), Lab 02 (Data Loading & Transformation), Lab 04 (Spark SQL & Optimization)

![Production Deployment Diagram](../assets/diagrams/lab-07-production-deployment.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
import os
import subprocess

In [ ]:
# Create Spark session with production configurations
spark = SparkSession.builder \
    .appName("ProductionDeployment") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "10") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.io.compression.codec", "snappy") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")
print(f"Dynamic allocation: {spark.conf.get('spark.dynamicAllocation.enabled')}")

## A. Docker Deployment

Containerize Spark applications for reproducible deployment.

In [ ]:
# Example Dockerfile for Spark application
dockerfile_content = '''
# Multi-stage build for optimized image size
FROM python:3.9-slim as builder

# Install build dependencies
RUN apt-get update && apt-get install -y gcc openjdk-11-jdk

# Copy requirements and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Runtime stage
FROM python:3.9-slim

# Install Java runtime
RUN apt-get update && apt-get install -y openjdk-11-jre && rm -rf /var/lib/apt/lists/*

# Copy Python dependencies from builder
COPY --from=builder /usr/local/lib/python3.9/site-packages /usr/local/lib/python3.9/site-packages

# Copy application code
COPY . /app
WORKDIR /app

# Set environment variables
ENV PYTHONPATH=/app
ENV SPARK_HOME=/opt/spark

# Run the application
CMD ["python", "main.py"]
'''

print("Dockerfile example created:")
print(dockerfile_content)

In [ ]:
# Example docker-compose.yml for Spark cluster
docker_compose_content = '''
version: '3.8'

services:
  spark-master:
    image: bitnami/spark:3.5.0
    container_name: spark-master
    ports:
      - "8080:8080"  # Spark Master UI
      - "7077:7077"  # Spark Master port
    environment:
      - SPARK_MODE=master
      - SPARK_MASTER_HOST=spark-master
      - SPARK_MASTER_PORT=7077
    volumes:
      - ./data:/opt/spark/data
      - ./logs:/opt/spark/logs
    networks:
      - spark-network

  spark-worker:
    image: bitnami/spark:3.5.0
    container_name: spark-worker
    depends_on:
      - spark-master
    environment:
      - SPARK_MODE=worker
      - SPARK_MASTER_HOST=spark-master
      - SPARK_MASTER_PORT=7077
      - SPARK_WORKER_CORES=2
      - SPARK_WORKER_MEMORY=4g
    volumes:
      - ./data:/opt/spark/data
      - ./logs:/opt/spark/logs
    networks:
      - spark-network

  spark-submit:
    build: .
    container_name: spark-submit
    depends_on:
      - spark-master
      - spark-worker
    environment:
      - SPARK_MASTER=spark://spark-master:7077
    volumes:
      - ./data:/app/data
      - ./logs:/app/logs
    networks:
      - spark-network

networks:
  spark-network:
    driver: bridge
'''

print("docker-compose.yml example created:")
print(docker_compose_content)

## B. Kubernetes Deployment

Deploy Spark applications on Kubernetes for scalability and high availability.

In [ ]:
# Example Kubernetes deployment for Spark application
k8s_deployment = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: spark-app
spec:
  replicas: 3
  selector:
    matchLabels:
      app: spark-app
  template:
    metadata:
      labels:
        app: spark-app
    spec:
      containers:
      - name: spark-app
        image: my-spark-app:latest
        ports:
          - containerPort: 4040
        env:
          - name: SPARK_MASTER
            value: "k8s://https://kubernetes.default.svc:443"
        resources:
          requests:
            memory: "2Gi"
            cpu: "1000m"
          limits:
            memory: "4Gi"
            cpu: "2000m"
        volumeMounts:
          - name: data
            mountPath: /data
      volumes:
        - name: data
          persistentVolumeClaim:
            claimName: spark-data-pvc
'''

print("Kubernetes deployment example created:")
print(k8s_deployment)

In [ ]:
# Example ConfigMap for Spark configuration
configmap_content = '''
apiVersion: v1
kind: ConfigMap
metadata:
  name: spark-config
data:
  spark-defaults.conf: |
    spark.driver.memory=2g
    spark.executor.memory=4g
    spark.executor.cores=2
    spark.sql.shuffle.partitions=200
    spark.dynamicAllocation.enabled=true
    spark.dynamicAllocation.minExecutors=1
    spark.dynamicAllocation.maxExecutors=10
    spark.serializer=org.apache.spark.serializer.KryoSerializer
    spark.io.compression.codec=snappy
'''

print("ConfigMap example created:")
print(configmap_content)

## C. Monitoring and Logging

Set up comprehensive monitoring and logging for production workloads.

In [ ]:
# Example Prometheus configuration for Spark metrics
prometheus_config = '''
global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: 'spark-master'
    static_configs:
      - targets: ['spark-master:8080']
    metrics_path: /metrics/prometheus/
    scrape_interval: 10s

  - job_name: 'spark-workers'
    static_configs:
      - targets: ['spark-worker-1:8081', 'spark-worker-2:8081']
    metrics_path: /metrics/prometheus/
    scrape_interval: 10s

  - job_name: 'spark-applications'
    static_configs:
      - targets: ['spark-submit:4040']
    metrics_path: /metrics/prometheus/
    scrape_interval: 10s
'''

print("Prometheus configuration example created:")
print(prometheus_config)

In [ ]:
# Example Grafana dashboard configuration
grafana_dashboard = '''
{
  "dashboard": {
    "title": "Spark Cluster Monitoring",
    "panels": [
      {
        "title": "Executor Memory Usage",
        "targets": [
          {
            "expr": "spark_executor_memory_used_bytes",
            "legendFormat": "{{executor}}"
          }
        ]
      },
      {
        "title": "Task Duration",
        "targets": [
          {
            "expr": "spark_task_duration_milliseconds",
            "legendFormat": "{{stage}}"
          }
        ]
      },
      {
        "title": "Shuffle Read/Write",
        "targets": [
          {
            "expr": "spark_shuffle_read_bytes_total",
            "legendFormat": "{{stage}}"
          }
        ]
      }
    ]
  }
}
'''

print("Grafana dashboard example created:")
print(grafana_dashboard)

## D. CI/CD Pipeline

Automate deployment with GitHub Actions.

In [ ]:
# Example GitHub Actions workflow
github_actions = '''
name: Build and Deploy Spark Application

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2

    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.9'

    - name: Install dependencies
      run: |
        pip install -r requirements.txt
        pip install pytest

    - name: Run tests
      run: pytest tests/

    - name: Build Docker image
      run: |
        docker build -t my-spark-app:${{ github.sha }} .
        docker tag my-spark-app:${{ github.sha }} my-spark-app:latest

    - name: Login to Docker Hub
      uses: docker/login-action@v1
      with:
        username: ${{ secrets.DOCKER_USERNAME }}
        password: ${{ secrets.DOCKER_PASSWORD }}

    - name: Push Docker image
      run: |
        docker push my-spark-app:${{ github.sha }}
        docker push my-spark-app:latest

  deploy:
    needs: build
    runs-on: ubuntu-latest
    environment:
      KUBECONFIG: ${{ secrets.KUBE_CONFIG }}

    steps:
    - uses: actions/checkout@v2

    - name: Set up kubectl
      uses: azure/setup-kubectl@v1

    - name: Deploy to Kubernetes
      run: |
        kubectl apply -f k8s/deployment.yaml
        kubectl apply -f k8s/service.yaml
        kubectl rollout status deployment/spark-app

    - name: Run smoke tests
      run: |
        kubectl exec -it deployment/spark-app -- python /app/tests/smoke_test.py
'''

print("GitHub Actions workflow example created:")
print(github_actions)

## E. Security Best Practices

Implement security measures for production deployments.

In [ ]:
# Example Kubernetes Secret for sensitive data
k8s_secret = '''
apiVersion: v1
kind: Secret
metadata:
  name: spark-secrets
type: Opaque
stringData:
  database-url: "jdbc:postgresql://db.example.com:5432/sparkdb"
  database-user: "spark_user"
  api-key: "your-api-key-here"
  encryption-key: "your-encryption-key"

---
apiVersion: v1
kind: Secret
metadata:
  name: spark-tls
type: kubernetes.io/tls
data:
  tls.crt: <base64-encoded-certificate>
  tls.key: <base64-encoded-key>
'''

print("Kubernetes Secret example created:")
print(k8s_secret)

In [ ]:
# Example RBAC configuration
rbac_config = '''
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: spark
rules:
- apiGroups: [""]
  resources: ["pods", "configmaps", "secrets"]
  verbs: ["get", "list", "create", "update"]

---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: spark-role-binding
  namespace: spark
subjects:
- kind: ServiceAccount
  name: spark-service-account
roleRef:
  kind: Role
  name: spark-role
  apiGroup: rbac.authorization.k8s.io
'''

print("RBAC configuration example created:")
print(rbac_config)

## F. Performance Optimization

Optimize Spark applications for production workloads.

In [ ]:
# Production Spark configuration for performance
performance_config = '''
# Memory configuration
spark.driver.memory=4g
spark.executor.memory=8g
spark.memory.fraction=0.6
spark.memory.storageFraction=0.5

# Parallelism
spark.default.parallelism=16
spark.sql.shuffle.partitions=200

# Adaptive Query Execution
spark.sql.adaptive.enabled=true
spark.sql.adaptive.coalescePartitions.enabled=true
spark.sql.adaptive.skewJoin.enabled=true

# Serialization
spark.serializer=org.apache.spark.serializer.KryoSerializer
spark.kryoserializer.buffer.max=256m

# Compression
spark.io.compression.codec=snappy
spark.rdd.compress=true
spark.sql.shuffle.compress=true

# Broadcast join
spark.sql.autoBroadcastJoinThreshold=10485760  # 10MB

# Speculation
spark.speculation=true
spark.speculation.multiplier=1.5

# Dynamic allocation
spark.dynamicAllocation.enabled=true
spark.dynamicAllocation.minExecutors=2
spark.dynamicAllocation.maxExecutors=20
spark.dynamicAllocation.initialExecutors=4
spark.dynamicAllocation.executorIdleTimeout=60s
spark.dynamicAllocation.schedulerBacklogTimeout=5s
'''

print("Production performance configuration:")
print(performance_config)

## G. High Availability

Configure high availability for production workloads.

In [ ]:
# High availability configuration
ha_config = '''
# Master HA with ZooKeeper
spark.deploy.recoveryMode=ZOOKEEPER
spark.deploy.zookeeper.url=zk1:2181,zk2:2181,zk3:2181

# Checkpointing for fault tolerance
spark.sql.streaming.checkpointLocation=/checkpoint/spark

# Write Ahead Logs for streaming
spark.sql.streaming.fileSink.log.compactWrites=true

# Enable retry logic
spark.task.maxFailures=4
spark.stage.maxConsecutiveAttempts=4

# Enable event log for recovery
spark.eventLog.enabled=true
spark.eventLog.dir=/eventlogs
spark.history.fs.logDirectory=/eventlogs
'''

print("High availability configuration:")
print(ha_config)

## Exam-Style Review

**Q1.** What is the advantage of using Kubernetes for Spark deployment?
- A) Easier local development
- B) Automatic scaling, self-healing, and resource management
- C) Lower cost only
- D) No configuration needed

**Q2.** What is the purpose of ConfigMaps in Kubernetes?
- A) To store secrets
- B) To store configuration data as key-value pairs
- C) To manage network policies
- D) To handle load balancing

**Q3.** What is Dynamic Allocation in Spark?
- A) Manual allocation of resources
- B) Automatic adjustment of executor count based on workload
- C) A type of data format
- D) A security feature

**Q4.** What is the purpose of checkpointing in Spark Streaming?
- A) To improve performance
- B) To provide fault tolerance and state recovery
- C) To compress data
- D) To monitor queries

### Answers
- **Q1: B** — Kubernetes provides automatic scaling, self-healing, and resource management for Spark deployments.
- **Q2: B** — ConfigMaps store configuration data as key-value pairs for applications.
- **Q3: B** — Dynamic Allocation automatically adjusts the number of executors based on workload demands.
- **Q4: B** — Checkpointing provides fault tolerance by periodically writing state to durable storage for recovery.

## Key Takeaways
- Docker provides reproducible deployment with containerization
- Kubernetes offers scalability, self-healing, and resource management
- ConfigMaps and Secrets manage configuration and sensitive data
- Prometheus and Grafana enable comprehensive monitoring
- CI/CD pipelines automate build, test, and deployment processes
- RBAC provides fine-grained access control
- Performance optimization requires proper configuration and tuning
- High availability requires checkpointing and master HA
- Security best practices include secret management and RBAC
- Monitoring and logging are essential for production operations

**Congratulations! You have completed all 7 labs of the Spark Code Practice repository.**

In [ ]:
# Cleanup
spark.catalog.clearCache()
print("Lab 07 complete. Cache cleared.")
print("\nProduction deployment examples saved for reference.")